# Experiment Analysis

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
%matplotlib notebook

In [ ]:
# Needed to import from the enderscope library
# RUN ONLY ONCE

import os

os.chdir("..")

## Imports

In [ ]:
from pathlib import Path

from tqdm import tqdm
from rich.pretty import pprint
from joblib import Parallel, delayed

import numpy as np
import pandas as pd
import cv2

import panel as pn

from enderleaf.const import COLOR_SPACES
from enderleaf.enums import ImageMergeMode, ImageMergeMethod
from enderleaf.draw import plot_images_with_histograms, concat_tile_resize
from enderleaf.tools import read_dataframe
from enderleaf.image import (
    load_image,
    crop_image,
    Rectangle,
    merge_images_channels,
    get_circles,
    to_pil
)

In [ ]:
pn.extension("ipywidgets")

## Constants

In [ ]:
EXP = "Exp26DM14"
# EXP = "Exp26DM01"
# INOC = "I1"
# PLATE = 0
# MONTH = 6
# DAY = 2

# PATTH_TO_ROOT = Path.home().joinpath("rclone_remotes", "tages")
# PATTH_TO_ROOT = Path("/").joinpath("Volumes", "LaCie", "enderleaf")
# PATTH_TO_ROOT = Path("/").joinpath("Volumes", "Samsung_T5", "Enderleaf")
PATTH_TO_ROOT = Path(".")

PATH_TO_DATA = PATTH_TO_ROOT.joinpath("output", "job_data", EXP)
PATH_TO_IMAGES = PATTH_TO_ROOT.joinpath("output", "images", EXP)

[p.is_dir() for p in [PATTH_TO_ROOT, PATH_TO_DATA, PATH_TO_IMAGES]]

## Load Data

In [ ]:
df = pd.DataFrame()
csv_files = [c for c in PATH_TO_DATA.rglob("*.csv") if c.name.startswith("._") is False]
print(len(csv_files))
good_ones = []
bad_ones = []
for f in tqdm(csv_files):
    try:
        df = pd.concat([df, read_dataframe(f)])
    except:
        bad_ones.append(f.resolve())
    else:
        good_ones.append(f.resolve())

print(len(good_ones))
print(len(bad_ones))

df = df.sort_values(["plate", "row", "col"]).dropna(subset="north")


def get_file_path(inoc, file_name):
    return PATH_TO_IMAGES.joinpath(inoc).joinpath(file_name).resolve()


def is_file_ok(file_path):
    return file_path.is_file() is True


df = df[df.exp == EXP]
df["file_path"] = df.apply(lambda x: get_file_path(x.inoc, x.file_name), axis=1)
df["card_count"] = df[["north", "east", "west", "south"]].astype(int).sum(axis=1)
df["file_ok"] = df.apply(lambda x: is_file_ok(x.file_path), axis=1)
print(df[df.file_ok == False].shape)
df = df[df.file_ok == True]
df["file_size"] = df["file_path"].apply(lambda x: x.stat().st_size)
df

In [ ]:
bad_ones

In [ ]:
def sizeof_fmt(num, suffix="B"):
    for unit in ("", "Ki", "Mi", "Gi", "Ti", "Pi", "Ei", "Zi"):
        if abs(num) < 1024.0:
            return f"{num:3.1f}{unit}{suffix}"
        num /= 1024.0
    return f"{num:.1f}Yi{suffix}"

def apply_szfmt(pd_series:pd.Series):
    return pd_series.apply(sizeof_fmt)

df.groupby(["date","inoc", "plate"]).agg(
    {"file_name": "count", "file_size": "sum"}
).assign(file_size=lambda x: apply_szfmt(x.file_size)).style.map(
    lambda x: f"color: red;" if x != 324 else "", subset=["file_name"]
)

In [ ]:
df["img_x_cycle"] =  df.groupby("cycle_id").inoc.transform("count")
df[df["img_x_cycle"] != 4]

In [ ]:
df.groupby(["inoc", "date"]).agg(
    {"file_name": "count", "file_size": "sum", "plate":"nunique"}
).assign(file_size=lambda x: apply_szfmt(x.file_size)).style

In [ ]:
def check_image(image_path):
    try:
        cv2.imread(str(image_path))
    except Exception as e:
        return f"Failed load image: {str(e)}"
    else:
        return None


data = Parallel(n_jobs=32)(
    delayed(check_image)(row.file_path) for row in tqdm(list(df.itertuples()))
)
print([d for d in data if d is not None])

In [ ]:
df.sort_values("file_size")

## Select Cycle ID

In [ ]:
def get_inocs(exp):
    return list(df[df.exp == exp].inoc.sort_values().unique())


def get_dates(exp, inoc):
    return list(df[(df.exp == exp) & (df.inoc == inoc)].date.sort_values().unique())


def get_plates(exp, inoc, date):
    return list(
        df[(df.exp == exp) & (df.inoc == inoc) & (df.date == date)]
        .plate.sort_values()
        .unique()
    )


In [ ]:
sel_exp = pn.widgets.Select(name="Experiment", options=list(df.exp.unique()), width=150)
sel_inoc = pn.widgets.Select(name="Inoc", options=get_inocs(sel_exp.value), width=150)
sel_date = pn.widgets.Select(
    name="Date",
    options=get_dates(sel_exp.value, sel_inoc.value),
    width=150,
)
sel_plate = pn.widgets.Select(
    name="Plate",
    options=get_plates(sel_exp.value, sel_inoc.value, sel_date.value),
    width=150,
)
sel_row = pn.widgets.Select(name="Row", options=list(df.row.unique()), width=100)
sel_col = pn.widgets.Select(name="Col", options=list(df.col.unique()), width=100)
ls_working = pn.indicators.LoadingSpinner(value=False, size=40, color="primary")

bt_random = pn.widgets.Button(name="Random Disc")

img_out = pn.pane.Image(
    sizing_mode="scale_width"
)  # pn.pane.Matplotlib(sizing_mode="scale_width")

updating = False


def filter_df() -> pd.DataFrame:
    return df[
        (df.exp == sel_exp.value)
        & (df.inoc == sel_inoc.value)
        & (df.date == sel_date.value)
        & (df.plate == sel_plate.value)
        & (df.col == sel_col.value)
        & (df.row == sel_row.value)
    ]


def on_random(event):
    global updating
    updating = True
    try:
        row = (
            df[["exp", "inoc", "date", "plate", "row", "col"]]
            .drop_duplicates()
            .sample(n=1)
            .iloc[0]
        )
        sel_exp.value = row.exp
        sel_inoc.value = row.inoc
        sel_date.value = row.date
        sel_plate.value = row.plate
        sel_row.value = row.row
    finally:
        updating = False
    sel_col.value = row.col


bt_random.on_click(on_random)


@pn.depends(sel_exp.param.value, watch=True)
def on_exp_changed(exp):
    sel_inoc.options = get_inocs(exp)
    sel_date.options = get_dates(exp, sel_inoc.value)
    sel_plate.options = get_plates(exp, sel_inoc.value, sel_date.value)


@pn.depends(sel_inoc.param.value, watch=True)
def on_inoc_changed(inoc):
    sel_date.options = get_dates(sel_exp.value, inoc)
    sel_plate.options = get_plates(sel_exp.value, inoc, sel_date.value)


@pn.depends(sel_date.param.value, watch=True)
def on_date_changed(date):
    sel_plate.options = get_plates(sel_exp.value, sel_inoc.value, date)


@pn.depends(
    *[
        w.param.value
        for w in [sel_exp, sel_inoc, sel_date, sel_plate, sel_row, sel_col]
    ],
    watch=True,
)
def on_ld_changed(exp, inoc, date, plate, row, col):
    global updating
    if updating is True:
        return
    updating = True
    ls_working.value = True
    try:
        merge_method = ImageMergeMethod.RGB
        df_ld = filter_df()
        first_image = load_image(df_ld.iloc[0].file_path)
        height, width, _ = first_image.shape
        circles = get_circles(
            first_image, color_space="hsv", channel="s", resize_factor=8
        )
        if len(circles["accepted"]) == 1:
            _, cx, cy, r = circles["accepted"][0]
            crop_data = Rectangle.from_circle((cx, cy, r + 16))
        else:
            crop_data = Rectangle(left=0, top=0, right=width, bottom=height)
        if len(df_ld.card_count.unique()) > 1:
            merged_images = []
            card_counts = []
            for card_count in df_ld.card_count.unique():
                image_list = [
                    crop_image(load_image(row[1].file_path), crop_data)
                    for row in df_ld[df_ld.card_count == card_count].iterrows()
                ]
                merged_images.append(
                    merge_images_channels(
                        image_list=image_list,
                        color_space=merge_method.value[0],
                        merge_modes=merge_method.value[1],
                    )
                )
                card_counts.append(f"{card_count} {len(image_list)}")
            img_out.object = plot_images_with_histograms(
                images=merged_images, color_spaces=["rgb"], titles=card_counts
            )
        else:
            image_list = [
                crop_image(load_image(row[1].file_path), crop_data)
                for row in df_ld.iterrows()
            ]
            image_list.append(
                merge_images_channels(
                    image_list=image_list,
                    color_space=merge_method.value[0],
                    merge_modes=merge_method.value[1],
                )
            )
            img_out.object = to_pil(
                concat_tile_resize([image_list[:2], image_list[2:4], [image_list[4]]])
            )
            # plot_images_with_histograms(
            #     images=image_list, color_spaces=["rgb"]
            # )
    finally:
        updating = False
        ls_working.value = False


on_ld_changed(
    sel_exp.value,
    sel_inoc.value,
    sel_date.value,
    sel_plate.value,
    sel_row.value,
    sel_col.value,
)


pn.Column(
    pn.FlexBox(
        sel_exp, sel_inoc, sel_date, sel_plate, sel_col, sel_row, bt_random, ls_working
    ),
    img_out,
)